# 7 — Surveillance et maintenance (CRISP-ML(Q))

Une phase de *surveillance et maintenance* est nécessaire, car un modèle en production se dégrade silencieusement quand le marché change. C'est l'objet de ce notebook.

**Le fil conducteur.** Les **trois couches de monitoring** ne sont pas une nouveauté : ce sont les **trois axes d'évaluation de la Phase 5** (mathématique, système, métier), projetés sur la production *en continu*. La Phase 5 a jugé le modèle une fois pour le **choisir** ; la Phase 7 rebranche les mêmes axes en permanence pour le **surveiller**. Chaque métrique est rattachée à un engagement du ML Canvas.

Ce notebook **ne réévalue pas** le champion (la **Phase 5 — NB4 — fait foi**, on ne la rejoue pas) : il définit le dispositif, calcule les éléments réellement mesurables aujourd'hui (PSI train/test, traçabilité MLflow des 17 modèles) et **référence** les analyses de la Phase 5 (équité OOF) plutôt que de les refaire.

In [1]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

import glob, math, shutil
import mlflow
from mlflow.tracking import MlflowClient

Dimensions brutes : (1460, 81)
Nombre de points supprimés : 2
Dimensions après suppression : (1458, 81)
Variables après ingénierie : 90 colonnes (+8 dérivées)
Corrélation des variables dérivées avec SalePrice_log :
  TotalSF            : +0.825
  TotalBathrooms     : +0.677
  HouseAge           : -0.588
  YearsSinceRemodel  : -0.569
  GarageAge          : -0.543
  HasGarage          : +0.323
  HasSecondFloor     : +0.151
  HasPool            : +0.077   <-- faible (|r| < 0.1)
X_train : (1166, 83)   |   X_test : (292, 83)
Colonnes retirées (colinéarité) : ['GarageArea', 'TotalBsmtSF', 'TotRmsAbvGrd', 'GarageYrBlt']
NaN dans X_train : 6276 cellules sur 96778
Audit de cardinalité des colonnes nominales (sur X_train) :
Colonne           modalités   bucket
  Neighborhood           25     TargetEncoder (haute)  [imput. mode]
  Exterior2nd            16     TargetEncoder (haute)  [imput. mode]
  Exterior1st            15     TargetEncoder (haute)  [imput. mode]
  MSSubClass             15     

## 7.1 Les trois couches de monitoring

**Le fil conducteur du livrable.** Les trois couches de monitoring **sont les trois axes d'évaluation de la Phase 5, projetés sur la production en continu** : la Phase 5 a jugé le modèle *une fois* (statique) pour le **choisir** ; la Phase 7 rebranche les mêmes axes *en permanence* pour le **surveiller**. Chaque métrique est rattachée à un engagement du ML Canvas.

### Couche 1 — Mathématique (le modèle se trompe-t-il davantage ?)
- **RMSLE sur prix réalisés** : recalculée dès qu'une vente se conclut, comparée au seuil **0,13** (NB4 §5.5.5).
- **Data drift** : PSI / KS sur les variables clés (`GrLivArea`, `TotalSF`, `Neighborhood`, `OverallQual`) entrée vs train.
- **Prediction drift** : dérive de la distribution des prix prédits.
- **Stabilité des importances SHAP** : un glissement du top-3 facteurs = signal de dérive de marché.
- **Déclencheurs de réentraînement** : cadence **mensuelle** *ou* RMSLE > 0,13 *ou* PSI au-delà d'un seuil.

### Couche 2 — Système / IT (l'API tient-elle ses promesses techniques ?)
- **Latence p50 / p95 / p99** vs SLA **5 s** (baseline de quelques dizaines de ms/requête mesurée, NB4 §5.5.3), mesurée sur les vraies requêtes HTTP via `mlflow models serve`.
- **Taux d'erreur** (5xx), **volume de requêtes**.
- **Logs de prédiction structurés**, rétention **90 jours** (alimente l'audit d'équité, couche 3).
- **Événements de déploiement** (`MlflowClient().search_model_versions`), **CPU / mémoire**.

### Couche 3 — Métier (l'outil crée-t-il la valeur promise ?)
- **KPIs du Canvas** : temps d'expertise **4 h → 2 h**, **+20 %** de ventes sous 90 jours, taux d'**override < 10 %**.
- **A/B test** de déploiement progressif : 3 mois, **50/50** vs expertise manuelle.
- **Audit d'équité géographique continu** (contrainte Canvas), la version production de l'analyse OOF de NB4 §5.5.4.
- **Boucle de feedback consultant** via CRM (capteur précoce de dérive, cf. §7.5) et **satisfaction client** post-vente.
- **Impact du label « Prix certifié par Inved AI »** sur la conversion et le prix de vente.

## 7.2 Traçabilité MLflow — l'enregistrement des expériences

Le registre construit en §6.3 contient le champion servi. Mais la **couche mathématique** du monitoring a besoin d'une **ligne de base** : les performances de *tous* les candidats, tracées de façon reproductible, pour mesurer la dérive run-over-run et documenter le choix du champion.

On réenregistre donc les **17 modèles** des quatre familles (lus depuis `results/family_*.json` + `results/preds_*.npz`, **sans réentraînement**) comme autant de *runs* de l'expérience `inved-house-price` : paramètres, métriques (RMSLE CV/holdout, temps d'ajustement) et figure prédit-vs-réel. C'est l'instantané de référence ; en production, chaque réentraînement mensuel ajoute une run, et comparer les RMSLE dans le temps *est* la détection de dérive de performance.

In [2]:
EXPERIMENT = 'inved-house-price'
mlflow.set_tracking_uri(f"file:{Path.cwd() / 'mlruns'}")
mlflow.set_experiment(EXPERIMENT)
client = MlflowClient()
exp = client.get_experiment_by_name(EXPERIMENT)

families = {
    'scaled':          ('results/family_scaled.json',          'results/preds_scaled.npz'),
    'sklearn_trees':   ('results/family_sklearn_trees.json',   'results/preds_sklearn_trees.npz'),
    'native_boosting': ('results/family_native_boosting.json', 'results/preds_native_boosting.npz'),
    'stacking':        ('results/family_stacking.json',        'results/preds_stacking.npz'),
}

# Idempotence : ne pas réenregistrer si l'instantané des 17 est déjà présent
existing = client.search_runs([exp.experiment_id], max_results=2000) if exp else []
existing_names = {r.data.tags.get('mlflow.runName') for r in existing}

logged = 0
for fam, (jf, npzf) in families.items():
    rows = json.loads(Path(jf).read_text())
    npz = np.load(npzf, allow_pickle=True)
    y_true = npz['y_true'].astype(float)
    for r in rows:
        if r['model'] == 'OLS_Baseline_5Feats':
            continue  # baseline metier (plancher) : pas de pred holdout publiee, hors suivi candidats
        if r['model'] in existing_names:
            continue
        with mlflow.start_run(run_name=r['model']):
            mlflow.log_param('family', r['family'])
            mlflow.log_param('model', r['model'])
            if r.get('params'):
                mlflow.log_param('params', json.dumps(r['params']))
            for metric in ('cv_rmsle', 'holdout_rmsle', 'fit_time_s'):
                v = r.get(metric)
                if v is not None and not (isinstance(v, float) and math.isnan(v)):
                    mlflow.log_metric(metric, float(v))
            yp = npz[r['model']].astype(float)
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.scatter(np.expm1(y_true), np.expm1(yp), s=8, alpha=0.4)
            lims = [float(np.expm1(y_true).min()), float(np.expm1(y_true).max())]
            ax.plot(lims, lims, 'r--', lw=1)
            ax.set_xlabel('Prix réel ($)'); ax.set_ylabel('Prix prédit ($)')
            ax.set_title(f"{r['model']} (holdout)")
            mlflow.log_figure(fig, 'pred_vs_actual.png')
            plt.close(fig)
            logged += 1

total = len(client.search_runs([exp.experiment_id], max_results=2000))
print(f"{logged} modèles enregistrés ; expérience '{EXPERIMENT}' = {total} runs (champion + 17 candidats).")

/home/bogomil/Documents/GitHub/predicting-house-prices-ml/.venv/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


17 modèles enregistrés ; expérience 'inved-house-price' = 18 runs (champion + 17 candidats).


> **Interface MLflow** — pour explorer les 18 runs visuellement (paramètres, métriques RMSLE, figures prédit-vs-réel) et le registre de modèles, lancer dans un terminal, à la racine du projet :
>
> ```bash
> mlflow ui --backend-store-uri file:./mlruns
> ```
>
> puis ouvrir [http://127.0.0.1:5000](http://127.0.0.1:5000) — onglets *Experiments* (`inved-house-price`) et *Models* (`inved-house-price@Production`).

## 7.3 Détection de dérive des données (data drift)

La couche mathématique surveille deux dérives. La **dérive des données** compare la distribution des variables d'entrée en production à celle d'entraînement, *indépendamment* de toute vente réalisée — c'est le signal le **plus précoce**. On la quantifie par le **PSI** (Population Stability Index) : découpage de chaque variable en déciles sur le train, puis écart de répartition des nouvelles données.

Convention : **PSI < 0,1** stable · **0,1–0,25** dérive modérée (à surveiller) · **> 0,25** dérive forte (réentraînement). Faute de flux de production, on l'illustre sur le **test Kaggle (1459 biens)** vu comme « données entrantes ». Le holdout interne partage par construction la distribution du train (PSI ≈ 0), il ne montrerait rien.

In [3]:
df_test = pd.read_csv('data/test.csv')
X_kaggle = engineer_features(df_test).drop(columns=['SalePrice', 'SalePrice_log', 'Id'], errors='ignore')
X_kaggle = X_kaggle[X.columns]

def psi(expected, actual, bins=10):
    cuts = np.unique(np.quantile(expected.dropna(), np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3:
        return np.nan
    e = np.histogram(expected.dropna(), bins=cuts)[0] / max(expected.notna().sum(), 1)
    a = np.histogram(actual.dropna(), bins=cuts)[0] / max(actual.notna().sum(), 1)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

num_feats = [c for c in ['GrLivArea', 'TotalSF', 'OverallQual', '1stFlrSF', 'LotArea',
                         'HouseAge', 'YearsSinceRemodel', 'GarageCars', 'TotalBathrooms', 'FullBath']
             if c in X.columns]
psi_df = (pd.DataFrame({'variable': num_feats,
                        'PSI': [round(psi(X_train[f], X_kaggle[f]), 4) for f in num_feats]})
          .sort_values('PSI', ascending=False).reset_index(drop=True))
psi_df['statut'] = pd.cut(psi_df['PSI'], [-1, 0.1, 0.25, np.inf],
                          labels=['stable', 'à surveiller', 'dérive forte'])
print("PSI train vs test Kaggle (variables clés) :")
display(psi_df)
print(f"\nMax PSI = {psi_df['PSI'].max():.4f} → " +
      ("aucune dérive notable (cohérent : même période, même marché Ames)."
       if psi_df['PSI'].max() < 0.1 else "dérive à investiguer."))

PSI train vs test Kaggle (variables clés) :


,variable,PSI,statut
0,GrLivArea,0.0241,stable
1,1stFlrSF,0.0140,stable
2,YearsSinceRemodel,0.0118,stable
3,TotalBathrooms,0.0113,stable
4,TotalSF,0.0092,stable
5,GarageCars,0.0078,stable
6,OverallQual,0.0059,stable
7,FullBath,0.0059,stable
8,HouseAge,0.0043,stable
9,LotArea,0.0042,stable



Max PSI = 0.0241 → aucune dérive notable (cohérent : même période, même marché Ames).


## 7.4 Politique de réentraînement

Le réentraînement est déclenché par **trois conditions, en OU** :

- **Cadence** — mensuelle, sur **fenêtre glissante** (engagement Canvas « Freshness by Retraining »), pour absorber la dérive lente du marché.
- **Performance** — RMSLE sur ventes réalisées **> 0,13** (seuil de déploiement, NB4 §5.5) : le modèle est passé sous le niveau livrable.
- **Dérive des données** — PSI **> 0,25** sur une variable clé (§7.3), même sans chute de RMSLE encore visible : alerte précoce.

Chaque réentraînement crée une **nouvelle version** dans le registre MLflow (§6.3) ; la promotion `@Production` n'a lieu qu'après le garde-fou de non-régression RMSLE (§6.4). Le *rollback* = repointer l'alias sur la version précédente, immédiat.

## 7.5 Cas d'école transverse : la gentrification

Ames abrite le campus de l'**Iowa State University** ; la pression étudiante peut **gentrifier** un quartier autrefois bon marché. C'est le scénario qui illustre le mieux pourquoi les **trois couches** sont nécessaires *ensemble*, il les frappe simultanément :

- **Couche 1 (mathématique) — double dérive.** *Data drift* : la distribution des prix d'un quartier se décale (PSI/KS sur `Neighborhood` × prix le détecte). *Concept drift* plus insidieux : la **relation features → prix change** (un même bien vaut soudain plus parce que le quartier est devenu prisé), la dérive de prédiction et la chute de RMSLE sur ventes réalisées le révèlent.
- **Couche 3 (métier) — signal précoce.** Les **consultants corrigent à la hausse** systématiquement dans ce quartier *avant* que les ventes réalisées ne confirment la tendance : le taux et le sens des *overrides* (suivis via le CRM) sont un **détecteur avancé**, plus rapide que la métrique mathématique qui attend les transactions.
- **Tension d'équité — inverse du redlining.** Le modèle, entraîné sur l'**historique**, **sous-estime** un quartier en gentrification (il n'a pas encore « vu » la hausse). Ici l'équité ne consiste pas à éviter de sur-pénaliser un quartier pauvre (redlining classique) mais à **ne pas figer** un quartier en mutation dans son passé, un biais temporel, pas géographique.

**Réponse outillée.** C'est exactement ce que couvre le dispositif : **fenêtre glissante mensuelle** + **déclencheur de réentraînement** (cadence *ou* RMSLE > 0,13 *ou* dérive PSI), et la boucle consultant comme capteur précoce. La gentrification n'est donc pas un angle mort, mais le **cas d'usage qui justifie** l'architecture de monitoring à trois couches.

## 7.6 Déploiement progressif et équité

**Déploiement progressif (A/B).** La mise en production suit un **rollout 50/50** sur **3 mois** (engagement Canvas) : la moitié des dossiers passe par l'IA, l'autre par l'expertise manuelle, pour mesurer en conditions réelles le gain de temps (4 h → 2 h), le taux d'override (< 10 %) et la conversion à 90 jours (+20 %) avant généralisation.

**Audit d'équité continu.** La contrainte du Canvas (pas de sous-évaluation géographique systématique, *anti-redlining*) est surveillée par l'analyse **OOF des résidus par quartier** établie en Phase 5 (**NB4 §5.5.4**) : médianes dans **≈ [−4 %, +6 %]**, sans biais directionnel d'ensemble, léger penchant à sous-estimer **BrkSide / IDOTRR (~+5–6 %)** sur effectifs suffisants, **signal mineur à suivre**, pas un motif de rejet. En production, ce même calcul tourne en continu sur les ventes réalisées ; un quartier qui décroche déclenche une investigation (lien avec le cas gentrification, §7.5). *(On référence ici le résultat de la Phase 5, sans le recalculer : la Phase 5 (NB4) fait foi.)*

## Transition — Phase 7 → Conclusion

Le dispositif est complet : déploiement (Phase 6), puis surveillance sur trois couches, détection de dérive, politique de réentraînement, rollout progressif et audit d'équité (Phase 7). Reste à prendre de la hauteur : la **conclusion** synthétise le parcours CRISP-ML(Q) de bout en bout pour la direction — du besoin métier au modèle livré, ses chiffres, ses limites et ses perspectives.